# 02 — Entrenamiento del Modelo de Detección de Fraude
**HackIAthon 2026 | Reto Aseguradora del Sur — FraudIA Claims**

Este notebook entrena el modelo de Machine Learning (XGBoost) para predecir la probabilidad de posible fraude en siniestros, usando la etiqueta simulada `etiqueta_fraude_simulada`.

**Enfoque híbrido**: Reglas de negocio (60%) + Predicción ML (40%) = Score final.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_score, recall_score, f1_score
)

try:
    from xgboost import XGBClassifier
    MODELO = 'XGBoost'
    print('✅ XGBoost disponible')
except ImportError:
    XGBClassifier = RandomForestClassifier
    MODELO = 'RandomForest (fallback)'
    print('⚠️  XGBoost no disponible, usando RandomForest')

sns.set_theme(style='whitegrid')
DATA_PATH  = Path('../backend/ai_data_core/data/synthetic/siniestros_scored.csv')
MODEL_PATH = Path('../backend/ai_data_core/data/processed/fraud_model.pkl')
ENC_PATH   = Path('../backend/ai_data_core/data/processed/label_encoders.pkl')
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Distribución etiqueta fraude: {df["etiqueta_fraude_simulada"].value_counts().to_dict()}')

## 1. Ingeniería de Features

In [ ]:
FEATURES_NUM = [
    'monto_reclamado', 'monto_estimado',
    'dias_desde_inicio_poliza', 'dias_desde_fin_poliza',
    'dias_entre_ocurrencia_reporte', 'historial_siniestros_asegurado',
    'score_riesgo',
    'tiene_doc_inconsistente',
]
FEATURES_CAT = ['ramo', 'cobertura', 'estado', 'sucursal']
TARGET = 'etiqueta_fraude_simulada'

df_ml = df.copy()

# Features derivadas
df_ml['suma_asegurada_proxy'] = df_ml['monto_reclamado'] / (df_ml['monto_estimado'] + 1)
df_ml['es_borde_vigencia']    = (df_ml['dias_desde_inicio_poliza'] <= 30).astype(int)
df_ml['reporte_tardio']       = (df_ml['dias_entre_ocurrencia_reporte'] > 7).astype(int)
df_ml['tiene_doc_inconsistente'] = df_ml['tiene_doc_inconsistente'].map(
    {True: 1, False: 0, 1: 1, 0: 0}
).fillna(0).astype(int)

FEATURES_NUM_EXTRA = FEATURES_NUM + ['suma_asegurada_proxy', 'es_borde_vigencia', 'reporte_tardio']

# Encoders para categóricas
encoders = {}
for col in FEATURES_CAT:
    le = LabelEncoder()
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col].astype(str).fillna('desconocido'))
    encoders[col] = le

FEATURES_FINAL = FEATURES_NUM_EXTRA + [c + '_enc' for c in FEATURES_CAT]
X = df_ml[FEATURES_FINAL].fillna(0)
y = df_ml[TARGET].astype(int)

print(f'Features totales: {len(FEATURES_FINAL)}')
print(f'Distribución Y → Fraude: {y.sum()} ({y.mean()*100:.1f}%) | Normal: {(1-y).sum()}')

## 2. Split Train / Test (80/20 estratificado)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras')
print(f'Fraudes en train: {y_train.sum()} | en test: {y_test.sum()}')

## 3. Entrenamiento del Modelo

In [ ]:
if MODELO == 'XGBoost':
    modelo = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        scale_pos_weight=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        verbosity=0,
    )
else:
    modelo = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight='balanced',
        random_state=42,
    )

modelo.fit(X_train, y_train)
print(f'✅ Modelo {MODELO} entrenado correctamente')

# Validación cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(modelo, X, y, cv=cv, scoring='roc_auc')
print(f'Cross-validation AUC-ROC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## 4. Importancia de Variables

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=FEATURES_FINAL)
importancias = importancias.sort_values(ascending=False)

plt.figure(figsize=(12, 6))
importancias.head(12).plot(kind='bar', color='#6366f1')
plt.title(f'Top 12 Features más importantes — {MODELO}', fontsize=14)
plt.xlabel('Feature')
plt.ylabel('Importancia')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 variables más predictivas:')
for feat, imp in importancias.head(5).items():
    print(f'  {feat}: {imp:.4f}')

## 5. Guardar Modelo

In [ ]:
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(modelo, f)
with open(ENC_PATH, 'wb') as f:
    pickle.dump(encoders, f)

print(f'✅ Modelo guardado en: {MODEL_PATH}')
print(f'✅ Encoders guardados en: {ENC_PATH}')

## 6. Notas del Modelo

- **Algoritmo**: XGBoost (con fallback a RandomForest si XGBoost no está instalado).
- **Etiqueta**: `etiqueta_fraude_simulada` — generada sintéticamente para entrenamiento.
- **Uso en producción**: El modelo genera una `probabilidad_fraude` [0.0-1.0].
- **Integración**: La API combina este score ML (40%) con el motor de reglas (60%) para el score final.
- **Limitaciones**: Entrenado con datos sintéticos; puede presentar falsos positivos. La revisión humana es obligatoria.

> ⚠️ Este modelo genera **alertas de revisión**, NO acusaciones automáticas de fraude.